# Decoding: From a Distribution to Text

The full code of [Decoding: From a Distribution to Text](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/decoding-from-logits-to-text) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/decoding-from-logits-to-text.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 20 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB). On a GPU other than the post's Apple M4, the 16-bit rounding differences in the batching experiment will differ. On an RTX 3090, 2 of the 20 prompts changed when batched, against 3 in the post.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 19 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "decoding-from-logits-to-text"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/decoding-from-logits-to-text


## `distributions.py`

Two real next-token distributions: the most likely tokens, the entropy at three temperatures, and how many tokens each filter keeps, before and after temperature. Writes `widget_dist.json`, the data for the post's widget.

In [2]:
!{sys.executable} distributions.py

'The KV cache stores the keys and' ->
   ' values' 0.757, ' their' 0.084, ' corresponding' 0.043, ' the' 0.015, ' associated' 0.008, ' data' 0.008


   entropy at T = 0.5 / 1 / 1.5: 0.09 / 1.41 / 6.11 nats;  top token's probability at T = 0.1: 1.0000


   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 5, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 3654, min-p 0.1 keeps 3; with the filters first, top-p's set stays 5 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 26093, min-p 0.1 keeps 6; with the filters first, top-p's set stays 5 and min-p's 2


'My favorite thing about this blog is' ->
   ' that' 0.460, ' the' 0.214, ' how' 0.039, ' when' 0.035, ' its' 0.014, ' seeing' 0.012
   entropy at T = 0.5 / 1 / 1.5: 0.58 / 2.52 / 6.91 nats;  top token's probability at T = 0.1: 0.9995
   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 26, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 9522, min-p 0.1 keeps 4; with the filters first, top-p's set stays 26 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 45599, min-p 0.1 keeps 20; with the filters first, top-p's set stays 26 and min-p's 2


the widget's 200 tokens cover 97.1% of the probability at T = 1; the other 151,736 tokens go in 87 bins


The post printed:

```text
'The KV cache stores the keys and' ->
   ' values' 0.758, ' their' 0.084, ' corresponding' 0.043, ' the' 0.015, ' associated' 0.008, ' data' 0.008
   entropy at T = 0.5 / 1 / 1.5: 0.09 / 1.40 / 6.11 nats;  top token's probability at T = 0.1: 1.0000
   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 5, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 3655, min-p 0.1 keeps 3; with the filters first, top-p's set stays 5 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 26113, min-p 0.1 keeps 6; with the filters first, top-p's set stays 5 and min-p's 2
'My favorite thing about this blog is' ->
   ' that' 0.461, ' the' 0.211, ' how' 0.039, ' when' 0.036, ' its' 0.014, ' seeing' 0.012
   entropy at T = 0.5 / 1 / 1.5: 0.57 / 2.53 / 6.92 nats;  top token's probability at T = 0.1: 0.9996
   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 26, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 9542, min-p 0.1 keeps 4; with the filters first, top-p's set stays 26 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 45600, min-p 0.1 keeps 20; with the filters first, top-p's set stays 26 and min-p's 2
the widget's 200 tokens cover 97.1% of the probability at T = 1; the other 151,736 tokens go in 87 bins
```

## `human_logprob.py`

The log-probability of each prompt's real continuation, and whether the paragraphs full of numbers are the hard ones.

In [3]:
!{sys.executable} human_logprob.py

per-prompt log-prob of the real continuation: -4.46, -4.36, -4.24, -4.18, -4.05, -4.03, -3.91, -3.87, -3.75, -3.56, -3.43, -3.35, -3.29, -3.05, -2.99, -2.96, -2.91, -2.90, -2.74, -2.49
lowest -4.46 (prompt 10, 0% of tokens with digits); highest -2.49 (prompt 17, 39% with digits); mean -3.53


The post printed:

```text
per-prompt log-prob of the real continuation: -4.46, -4.36, -4.24, -4.18, -4.05, -4.03, -3.91, -3.87, -3.75, -3.56, -3.43, -3.35, -3.29, -3.05, -2.99, -2.96, -2.91, -2.90, -2.74, -2.49
lowest -4.46 (prompt 10, 0% of tokens with digits); highest -2.49 (prompt 17, 39% with digits); mean -3.53
```

## `compare.py`

Every policy on the 20 prompts, 128 new tokens each, one or two seeds: repeated 4-grams and log-probability, against the real continuations. Writes `compare_texts.json`.

In [4]:
!{sys.executable} compare.py

greedy                   seed 0: first 64 tokens: repeated 4-grams 24.8%, log-prob -0.85; all 128: repeated 49.1%, 14/20 over 30%  (62 s)


beam search, 4 beams     seed 0: first 64 tokens: repeated 4-grams 15.9%, log-prob -0.63; all 128: repeated 37.6%, 12/20 over 30%  (66 s)


greedy + repetition 1.2  seed 0: first 64 tokens: repeated 4-grams  1.0%, log-prob -1.43; all 128: repeated  2.6%,  0/20 over 30%  (61 s)


pure sampling, T=1       seed 0: first 64 tokens: repeated 4-grams  1.2%, log-prob -3.51; all 128: repeated  1.5%,  0/20 over 30%  (60 s)


pure sampling, T=1       seed 1: first 64 tokens: repeated 4-grams  2.8%, log-prob -2.81; all 128: repeated  3.2%,  0/20 over 30%  (60 s)


...

T=1.5 alone              seed 0: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.12; all 128: repeated  0.0%,  0/20 over 30%  (60 s)


T=1.5 alone              seed 1: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.02; all 128: repeated  0.0%,  0/20 over 30%  (60 s)


T=1.5 + min-p 0.1        seed 0: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.15; all 128: repeated  1.6%,  0/20 over 30%  (58 s)


T=1.5 + min-p 0.1        seed 1: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.14; all 128: repeated  3.8%,  0/20 over 30%  (59 s)


human continuation             : first 64 tokens: repeated 4-grams  0.3%, log-prob -3.53


The post printed:

```text
greedy                   seed 0: first 64 tokens: repeated 4-grams 25.0%, log-prob -0.85; all 128: repeated 48.6%, 13/20 over 30%  (124 s)
beam search, 4 beams     seed 0: first 64 tokens: repeated 4-grams 16.8%, log-prob -0.62; all 128: repeated 41.4%, 14/20 over 30%  (131 s)
greedy + repetition 1.2  seed 0: first 64 tokens: repeated 4-grams  1.0%, log-prob -1.43; all 128: repeated  2.6%,  0/20 over 30%  (107 s)
pure sampling, T=1       seed 0: first 64 tokens: repeated 4-grams  0.5%, log-prob -3.15; all 128: repeated  1.8%,  0/20 over 30%  (105 s)
pure sampling, T=1       seed 1: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.86; all 128: repeated  3.5%,  1/20 over 30%  (101 s)
T=0.7                    seed 0: first 64 tokens: repeated 4-grams  3.4%, log-prob -1.46; all 128: repeated  5.8%,  1/20 over 30%  (109 s)
T=0.7                    seed 1: first 64 tokens: repeated 4-grams  3.8%, log-prob -1.66; all 128: repeated  8.1%,  2/20 over 30%  (112 s)
top-k 40                 seed 0: first 64 tokens: repeated 4-grams  1.2%, log-prob -2.10; all 128: repeated  2.0%,  0/20 over 30%  (113 s)
top-k 40                 seed 1: first 64 tokens: repeated 4-grams  3.4%, log-prob -1.93; all 128: repeated  5.0%,  0/20 over 30%  (107 s)
top-p 0.9                seed 0: first 64 tokens: repeated 4-grams  0.3%, log-prob -2.30; all 128: repeated  2.4%,  0/20 over 30%  (119 s)
top-p 0.9                seed 1: first 64 tokens: repeated 4-grams  3.9%, log-prob -2.11; all 128: repeated  7.0%,  1/20 over 30%  (112 s)
min-p 0.1                seed 0: first 64 tokens: repeated 4-grams  3.3%, log-prob -1.49; all 128: repeated  9.2%,  2/20 over 30%  (106 s)
min-p 0.1                seed 1: first 64 tokens: repeated 4-grams  2.9%, log-prob -1.34; all 128: repeated  8.4%,  1/20 over 30%  (113 s)
T=1.5 alone              seed 0: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.13; all 128: repeated  0.0%,  0/20 over 30%  (111 s)
T=1.5 alone              seed 1: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.24; all 128: repeated  0.0%,  0/20 over 30%  (111 s)
T=1.5 + min-p 0.1        seed 0: first 64 tokens: repeated 4-grams  2.4%, log-prob -2.18; all 128: repeated  4.2%,  1/20 over 30%  (125 s)
T=1.5 + min-p 0.1        seed 1: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.06; all 128: repeated  3.7%,  0/20 over 30%  (120 s)
human continuation             : first 64 tokens: repeated 4-grams  0.3%, log-prob -3.53
```

## `one_prompt.py`

One prompt's continuation under each policy, from `compare_texts.json`, with the real continuation last.

In [5]:
!{sys.executable} one_prompt.py

PROMPT: All three need about the same number of updates to get close, around 50. But an update on one question costs a 24th of a full-batch update. So SGD gets there after seeing the data about twice, where

[greedy]" as the other two need to see the data about 10 times. \n\nSo, the question is, why does SGD need to see the data about 10 times, while the other two need to see the data about 50 times? \n\nI understand that SGD is a stochastic algorithm, but I don't understand w"

[beam search, 4 beams]' mini-batch gets there after seeing the data about 1.5 times. This is because mini-batch has to see the data 1.5 times to get the same number of updates as SGD, and SGD has to see the data 2 times to get the same number of updates as mini-batch. So mini-batch '

[greedy + repetition 1.2]" as the other two need to see it 3 times. I'm not sure if this is a good thing or bad, but my intuition says that SGD should be faster than both of them.\n\nI've been trying to understand why and how thes

The post printed:

```text
PROMPT: All three need about the same number of updates to get close, around 50. But an update on one question costs a 24th of a full-batch update. So SGD gets there after seeing the data about twice, where

[greedy]" as the other two need to see the data about 10 times. \n\nSo, the question is, why does SGD need to see the data about 10 times, while the other two need to see the data about 50 times? \n\nI understand that SGD is a stochastic algorithm, but I don't understand w"

[beam search, 4 beams]' mini-batch gets there after seeing the data about 1.5 times. This is because mini-batch has to see the data 1.5 times to get the same number of updates as SGD, and SGD has to see the data 2 times to get the same number of updates as mini-batch. So mini-batch '

[greedy + repetition 1.2]" as the other two need to see it 3 times. I'm not sure if this is a good thing or bad, but my intuition says that SGD should be faster than both of them.\n\nI've been trying to understand why and how these things work in practice (and what they mean), so any hel"

[pure sampling, T=1]' agrad and rose have each seen almost the entire batch of updates. Worst case, I suppose. So far, by updating the golden set, I have trouble seeing a good upper bound on the number of updates it takes. Can anyone help with referencing the complexities to get t'

[top-p 0.9]' as Batch would take 15 minutes for a full-batch. \\n\\nA Batch update with a full-batch would result in a new estimate for the error after every update. If, instead, the batch update with a partial batch is used, then the error can only be updated after every $'

[min-p 0.1]' as Adam and RMSProp each take about the same amount of updates. \\n\\n'

[T=1.5 alone]' agodly rose closes in without using the heavy Smith Archive warmup quadrant.\\nTS_USD_OPEN安い decimal by:", finelist الأمم synchronization method with trouble�iers SMART công&PubOrganwoo-oriented cetверParser).\\-output[i referencing執INCLUDINGCKكف纂-language sub_'

[T=1.5 + min-p 0.1]' SGA would have seen it almost four times over.\\nI can tell you the size of my learning batches and the total number of updates needed, so you could simulate your own. Would the algorithm speed up more quickly than it would if the batch size and total updates '

[what the post says]" full-batch descent needs 52 passes. The price is noise, and it shows in the last two columns. With a constant learning rate, SGD never settles. After 5,000 updates its median loss is 0.4949 against the optimum's 0.4837, and its worst run is at 0.7688, knocked"
```

## `reproducible.py`

The same seed twice, and greedy decoding alone against in a padded batch of four.

In [6]:
!{sys.executable} reproducible.py

top-p 0.9, the same seed twice: identical text for 5 of 5 prompts


greedy, alone vs in a padded batch of 4: largest first-step logit difference 0.035 (median 0.027); the 64-token outputs differ for 2 of 20 prompts, first at token [52, 63]


The post printed:

```text
top-p 0.9, the same seed twice: identical text for 5 of 5 prompts
greedy, alone vs in a padded batch of 4: largest first-step logit difference 0.039 (median 0.027); the 64-token outputs differ for 3 of 20 prompts, first at token [22, 37, 52]
```

## `det_check_both.py`

Runs `det_check.py 16` and `det_check.py 32`.

In [7]:
!{sys.executable} det_check_both.py

16 0 pad tokens 17 diff 0.0312 with position_ids 0.0234


16 1 pad tokens 7 diff 0.0293 with position_ids 0.0273
16 2 pad tokens 0 diff 0.0234 with position_ids 0.0234


16 3 pad tokens 16 diff 0.0273 with position_ids 0.0273


32 0 pad tokens 17 diff 0.0 with position_ids 0.0


32 1 pad tokens 7 diff 0.0 with position_ids 0.0
32 2 pad tokens 0 diff 0.0 with position_ids 0.0


32 3 pad tokens 16 diff 0.0 with position_ids 0.0


The post printed:

```text
16 0 pad tokens 17 diff 0.0195 with position_ids 0.0312
16 1 pad tokens 7 diff 0.0234 with position_ids 0.0234
16 2 pad tokens 0 diff 0.0273 with position_ids 0.0273
16 3 pad tokens 16 diff 0.0293 with position_ids 0.0352
32 0 pad tokens 17 diff 0.0 with position_ids 0.0
32 1 pad tokens 7 diff 0.0 with position_ids 0.0
32 2 pad tokens 0 diff 0.0 with position_ids 0.0
32 3 pad tokens 16 diff 0.0001 with position_ids 0.0001
```

## `cost.py`

One forward pass for a token against the time to choose it under three policies.

In [8]:
!{sys.executable} cost.py

forward pass for one token: 23.2 ms
choosing the token, greedy: 0.06 ms


choosing the token, top-p 0.9: 0.40 ms


choosing the token, T 0.7 + top-k 40 + top-p 0.9 + min-p 0.05 + repetition 1.1: 0.82 ms


The post printed:

```text
forward pass for one token: 41.0 ms
choosing the token, greedy: 0.24 ms
choosing the token, top-p 0.9: 0.91 ms
choosing the token, T 0.7 + top-k 40 + top-p 0.9 + min-p 0.05 + repetition 1.1: 1.61 ms
```